<a href="https://colab.research.google.com/github/kokila83/AgenticAIMani_SampleMiniProject/blob/main/Ex%201_Function_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client with your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Tool definition for the model
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["city"]
            }
        }
    }
]

# 4. Actual local Python implementation
def get_weather(city, unit="celsius"):
    return f"Weather in {city}: 28°{unit[0].upper()}, Sunny"

# 5. Call Azure AI Foundry with tool declaration
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What is the weather in Mumbai?"}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

# 6. Process the tool call if model requests it
if message.tool_calls:
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_weather":
            args = json.loads(tool_call.function.arguments)
            result = get_weather(**args)
            print(f"Agent executed tool call -> {result}")
else:
    print(message.content)

Enter your Azure AI Foundry API key: ··········
Agent executed tool call -> Weather in Mumbai: 28°C, Sunny


In [8]:
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Define tools using standard OpenAI tool-calling format
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get current stock price",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "Stock symbol (e.g. RELIANCE)"
                    }
                },
                "required": ["symbol"]
            }
        }
    }
]

# 4. Dummy stock price implementation function
def get_stock_price(symbol):
    return f"The current price for {symbol} is ₹2,980.50"

# 5. Send request with tools parameter
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What's Reliance stock price?"}],
    tools=tools,
    tool_choice="auto"
)

# 6. Parse and execute the tool call
message = response.choices[0].message

if message.tool_calls:
    for tool_call in message.tool_calls:
        print(f"Function requested: {tool_call.function.name}")
        args = json.loads(tool_call.function.arguments)
        print(f"Args: {args}")

        # Execute local tool if matched
        if tool_call.function.name == "get_stock_price":
            result = get_stock_price(**args)
            print(f"Result: {result}")
else:
    print(f"Response: {message.content}")

Enter your Azure AI Foundry API key: ··········
Function requested: get_stock_price
Args: {'symbol': 'RELIANCE'}
Result: The current price for RELIANCE is ₹2,980.50


In [9]:
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. IT Helpdesk Tool Implementations
def reset_password(employee_id):
    return f"Password reset for {employee_id}. New temp password sent to email."

def check_vpn_status(employee_id):
    return f"VPN Status for {employee_id}: Connected, Last active 2 mins ago"

def create_ticket(issue, priority):
    return f"Ticket #12345 created: {issue} (Priority: {priority})"

tools_map = {
    "reset_password": reset_password,
    "check_vpn_status": check_vpn_status,
    "create_ticket": create_ticket
}

# 4. IT Helpdesk Agent Runner
def helpdesk_agent(user_query):
    prompt = f"""You are an IT Helpdesk Agent. Available Tools:
    - reset_password(employee_id)
    - check_vpn_status(employee_id)
    - create_ticket(issue, priority)

    User Query: {user_query}

    Respond ONLY with valid JSON in this exact structure:
    {{"tool": "name", "args": {{"param": "value"}}}}"""

    try:
        # Call Azure AI Foundry API
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        raw_output = response.choices[0].message.content.strip()
        decision = json.loads(raw_output)

        tool_name = decision.get("tool")
        args = decision.get("args", {})

        if tool_name in tools_map:
            result = tools_map[tool_name](**args)
            print(f"🔧 Executed: {tool_name}")
            print(f"📋 Result: {result}\n")
            return result
        else:
            print(f"Response: {raw_output}\n")
            return raw_output

    except json.JSONDecodeError:
        print(f"Error parsing JSON. Raw output: {response.choices[0].message.content}")
    except Exception as e:
        print(f"Error executing agent: {e}\n")

# 5. Run test queries
if __name__ == "__main__":
    helpdesk_agent("I forgot my password, my ID is EMP123")
    helpdesk_agent("Is my VPN working? ID: EMP456")

Enter your Azure AI Foundry API key: ··········
🔧 Executed: reset_password
📋 Result: Password reset for EMP123. New temp password sent to email.

🔧 Executed: check_vpn_status
📋 Result: VPN Status for EMP456: Connected, Last active 2 mins ago



In [10]:
#Ex 4: Tool Chains (Multi-step)
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Define local tool functions
def search_flights(origin, destination, date):
    return [{"flight": "AI101", "price": 5000}, {"flight": "6E201", "price": 4500}]

def book_flight(flight_id, passenger):
    return {"booking_id": "BK123", "flight": flight_id, "passenger": passenger}

def send_confirmation(email, booking_id):
    return f"Confirmation sent to {email} for {booking_id}"

tools = {
    "search_flights": search_flights,
    "book_flight": book_flight,
    "send_confirmation": send_confirmation
}

# 4. Multi-step Travel Agent loop
def travel_agent(goal, user_info):
    context = f"User info: {user_info}"
    history = []

    for step in range(5):
        prompt = f"""Travel booking agent.
Goal: {goal}
{context}
Previous steps executed: {json.dumps(history)}

Tools:
- search_flights(origin, destination, date)
- book_flight(flight_id, passenger)
- send_confirmation(email, booking_id)

Respond ONLY in valid JSON using one of these forms:
{{"tool": "name", "args": {{...}}}}
OR
{{"done": true, "summary": "final confirmation message"}}"""

        try:
            # Call Azure AI Foundry model
            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[{"role": "user", "content": prompt}]
            )

            raw_content = response.choices[0].message.content.strip()
            action = json.loads(raw_content)

            # Check if workflow is complete
            if action.get("done"):
                print(f"\n✅ Final Summary: {action['summary']}")
                break

            # Execute selected tool
            tool_name = action.get("tool")
            args = action.get("args", {})

            if tool_name in tools:
                result = tools[tool_name](**args)
                history.append({"tool": tool_name, "result": result})
                print(f"Step {step + 1}: Executed '{tool_name}' -> {result}")
            else:
                print(f"Step {step + 1}: Unrecognized tool '{tool_name}' requested.")
                break

        except json.JSONDecodeError:
            print(f"Failed to parse JSON output: {response.choices[0].message.content}")
            break
        except Exception as e:
            print(f"Execution Error: {e}")
            break

# 5. Run the Travel Agent
if __name__ == "__main__":
    travel_agent(
        goal="Book cheapest Mumbai to Delhi flight for tomorrow",
        user_info={"name": "Rahul", "email": "rahul@email.com"}
    )

Enter your Azure AI Foundry API key: ··········
Step 1: Executed 'search_flights' -> [{'flight': 'AI101', 'price': 5000}, {'flight': '6E201', 'price': 4500}]
Step 2: Executed 'book_flight' -> {'booking_id': 'BK123', 'flight': '6E201', 'passenger': {'name': 'Rahul'}}
Step 3: Executed 'send_confirmation' -> Confirmation sent to rahul@email.com for BK123

✅ Final Summary: Flight 6E201 from Mumbai to Delhi for tomorrow has been booked at the cheapest price of 4500. Confirmation has been sent to rahul@email.com.
